# Stage 2 — Utilities: Shared Dataset Loader, Primary 30-Day Cohort Loader, and Model Pipeline

**Module Role (`src/utils`):** Central reference for shared project paths (`data/processed/`, `results/`, `figures/`), loading `dataset1_storyteller.csv` and `dataset2_ecog.csv`, loading (or deterministically building) the single primary 30-day matched cohort (`primary_multimodal_30day_cohort.csv`, 77 assessments / 71 participants), and defining the canonical `LogisticRegression` pipeline, participant-level aggregation, and metric functions.


## 1. Load source datasets and configure output folders


In [10]:
# Load source processed datasets (Storyteller + ECog-12 PT) and configure output folders.
# The primary 30-day matched multimodal cohort (77 assessments / 71 participants) is built
# and saved in 03_train_temporal_and_ablation.ipynb as primary_multimodal_30day_cohort.csv.

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = None
DATA_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (folder / "data" / "processed", folder / "processed_data"):
        if candidate.is_dir() and all(
            (candidate / filename).is_file()
            for filename in (
                "dataset1_storyteller.csv",
                "dataset2_ecog.csv",
            )
        ):
            PROJECT_DIR = folder
            DATA_DIR = candidate
            break
    if DATA_DIR is not None:
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data/processed/ containing dataset1_storyteller.csv and dataset2_ecog.csv."
    )

RESULTS_ROOT = PROJECT_DIR / "results"
FIGURES_ROOT = PROJECT_DIR / "figures"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
PRIMARY_COHORT_CSV = DATA_DIR / "primary_multimodal_30day_cohort.csv"

dataset1_storyteller = pd.read_csv(DATA_DIR / "dataset1_storyteller.csv", low_memory=False)
dataset2_ecog = pd.read_csv(DATA_DIR / "dataset2_ecog.csv", low_memory=False)

# Normalize identifiers, targets, and dates without changing cohort membership.
for df in (dataset1_storyteller, dataset2_ecog):
    if "PTID" not in df or "TARGET" not in df:
        raise KeyError("Each dataset must contain PTID and TARGET columns.")
    df["PTID"] = df["PTID"].astype("string").str.strip()
    df["TARGET"] = pd.to_numeric(df["TARGET"], errors="coerce")
    if df[["PTID", "TARGET"]].isna().any().any():
        raise ValueError("A dataset contains missing PTID or TARGET values.")
    if not df["TARGET"].isin([0, 1]).all():
        raise ValueError("TARGET must contain only 0 and 1.")
    df["TARGET"] = df["TARGET"].astype(int)
    for col in df.columns:
        if "DATE" in str(col).upper():
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Apply the feature exclusions used in the predictor audit (51 Storyteller, 12 ECOG).
STORY_EXCLUDED_TERMS = ("PLAYBACK_COUNT", "GMATCH", "LANGUAGE")
story_feature_cols = [
    col for col in dataset1_storyteller.columns
    if str(col).upper().startswith("STORY")
    and not any(term in str(col).upper() for term in STORY_EXCLUDED_TERMS)
]
for col in story_feature_cols:
    dataset1_storyteller[col] = pd.to_numeric(dataset1_storyteller[col], errors="coerce")
story_feature_cols = [
    col for col in story_feature_cols
    if dataset1_storyteller[col].notna().any()
]

ecog_feature_cols = [f"ECOGPT{i}" for i in range(1, 13)]
missing_ecog = [col for col in ecog_feature_cols if col not in dataset2_ecog.columns]
if missing_ecog:
    raise KeyError(f"Dataset 2 is missing predictor columns: {missing_ecog}")
for col in ecog_feature_cols:
    dataset2_ecog[col] = pd.to_numeric(dataset2_ecog[col], errors="coerce")

print("Loaded source datasets from:", DATA_DIR)
for name, df in (
    ("Storyteller Source Cohort", dataset1_storyteller),
    ("ECog-12 PT Source Cohort", dataset2_ecog),
):
    print(
        f"{name}: {len(df)} rows, {df['PTID'].nunique()} participants, "
        f"targets={df['TARGET'].value_counts().sort_index().to_dict()}"
    )
print(
    f"Features: {len(story_feature_cols)} Storyteller, "
    f"{len(ecog_feature_cols)} ECog-12 PT, "
    f"{len(story_feature_cols) + len(ecog_feature_cols)} full fusion"
)


Loaded source datasets from: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed
Storyteller Source Cohort: 1430 rows, 490 participants, targets={0: 1073, 1: 357}
ECog-12 PT Source Cohort: 501 rows, 397 participants, targets={0: 273, 1: 228}
Features: 51 Storyteller, 12 ECog-12 PT, 63 full fusion


C:\Users\amnar\AppData\Local\Temp\ipykernel_93128\1372921868.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


## 2. Load primary 30-day multimodal cohort and build feature matrices


In [8]:
# Load the single primary 30-day multimodal cohort saved by 03_train_temporal_and_ablation.ipynb
# (or build it deterministically from dataset1_storyteller + dataset2_ecog if not yet on disk).

STORY_DATE = "EXAMDATE_STORY"
ECOG_DATE = "COLDATE"
ID_COL = "PTID"
TARGET_COL = "TARGET"

if PRIMARY_COHORT_CSV.is_file():
    primary_multimodal_30day_cohort = pd.read_csv(PRIMARY_COHORT_CSV, low_memory=False)
    primary_multimodal_30day_cohort[ID_COL] = (
        primary_multimodal_30day_cohort[ID_COL].astype("string").str.strip()
    )
    primary_multimodal_30day_cohort[TARGET_COL] = (
        pd.to_numeric(primary_multimodal_30day_cohort[TARGET_COL], errors="raise").astype(int)
    )
    if STORY_DATE in primary_multimodal_30day_cohort.columns:
        primary_multimodal_30day_cohort[STORY_DATE] = pd.to_datetime(
            primary_multimodal_30day_cohort[STORY_DATE], errors="coerce"
        )
    cohort_source_label = f"saved CSV ({PRIMARY_COHORT_CSV.name})"
else:
    story = dataset1_storyteller.copy()
    ecog = dataset2_ecog.copy()
    story[STORY_DATE] = pd.to_datetime(story[STORY_DATE], errors="coerce")
    ecog[ECOG_DATE] = pd.to_datetime(ecog[ECOG_DATE], errors="coerce")
    story = story.dropna(subset=[ID_COL, STORY_DATE, TARGET_COL]).copy()
    ecog = ecog.dropna(subset=[ID_COL, ECOG_DATE]).copy()
    story[TARGET_COL] = pd.to_numeric(story[TARGET_COL], errors="coerce")
    story = story[story[TARGET_COL].isin([0, 1])].copy()

    for col in ecog_feature_cols:
        ecog[col] = pd.to_numeric(ecog[col], errors="coerce").replace(9, np.nan)
    ecog = ecog[ecog[ecog_feature_cols].notna().any(axis=1)].copy()
    for col in story_feature_cols:
        story[col] = pd.to_numeric(story[col], errors="coerce")

    story = story.reset_index(drop=True)
    ecog = ecog.reset_index(drop=True)
    ecog_by_id = {
        pid: group.sort_values(ECOG_DATE).copy()
        for pid, group in ecog.groupby(ID_COL)
    }

    records = []
    eligible_indices = []
    for idx, row in story.iterrows():
        pid = row[ID_COL]
        story_date = row[STORY_DATE]
        participant_ecog = ecog_by_id.get(pid)
        if participant_ecog is None:
            continue
        eligible = participant_ecog[
            (participant_ecog[ECOG_DATE] <= story_date)
            & ((story_date - participant_ecog[ECOG_DATE]).dt.days <= 30)
        ].copy()
        if eligible.empty:
            continue
        eligible["GAP_DAYS"] = (story_date - eligible[ECOG_DATE]).dt.days
        eligible_indices.append(idx)
        aggregated = eligible[ecog_feature_cols].median()
        rec = {
            "ROW_ID": len(records),
            ID_COL: pid,
            TARGET_COL: int(row[TARGET_COL]),
            "N_ECOG_RECORDS": len(eligible),
            "LATEST_ECOG_GAP": eligible["GAP_DAYS"].min(),
            "OLDEST_ECOG_GAP": eligible["GAP_DAYS"].max(),
            STORY_DATE: story_date,
        }
        rec.update(aggregated.to_dict())
        records.append(rec)

    common_story = story.loc[eligible_indices].copy().reset_index(drop=True)
    ecog_30d = pd.DataFrame(records)
    primary_multimodal_30day_cohort = pd.concat(
        [
            ecog_30d[["ROW_ID", ID_COL, TARGET_COL, "N_ECOG_RECORDS", "LATEST_ECOG_GAP", "OLDEST_ECOG_GAP", STORY_DATE]],
            common_story[story_feature_cols].apply(pd.to_numeric, errors="coerce").reset_index(drop=True),
            ecog_30d[ecog_feature_cols].apply(pd.to_numeric, errors="coerce").reset_index(drop=True),
        ],
        axis=1,
    )
    primary_multimodal_30day_cohort.to_csv(PRIMARY_COHORT_CSV, index=False)
    cohort_source_label = f"built from source CSVs and saved to {PRIMARY_COHORT_CSV.name}"

# Build the exact feature matrices consumed by Studies 7-13
X_story22 = (
    primary_multimodal_30day_cohort[story_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True)
)
X_ecog22 = (
    primary_multimodal_30day_cohort[ecog_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True)
)
X_full22 = pd.concat([X_story22, X_ecog22], axis=1)
y22 = primary_multimodal_30day_cohort[TARGET_COL].astype(int).to_numpy()
groups22 = primary_multimodal_30day_cohort[ID_COL].to_numpy()

ecog_without_1 = [col for col in ecog_feature_cols if col != "ECOGPT1"]
ecog_without_2 = [col for col in ecog_feature_cols if col != "ECOGPT2"]
ecog_without_1_2 = [col for col in ecog_feature_cols if col not in ["ECOGPT1", "ECOGPT2"]]

candidate_features22 = {
    "Full_fusion": list(story_feature_cols) + list(ecog_feature_cols),
    "Without_ECOGPT1": list(story_feature_cols) + ecog_without_1,
    "Without_ECOGPT2": list(story_feature_cols) + ecog_without_2,
    "Without_ECOGPT1_ECOGPT2": list(story_feature_cols) + ecog_without_1_2,
}

print(
    f"Primary 30-day cohort ready [{cohort_source_label}]: "
    f"{len(primary_multimodal_30day_cohort)} assessments, "
    f"{pd.Series(groups22).nunique()} participants "
    f"(CN={int((y22 == 0).sum())}, MCI={int((y22 == 1).sum())})"
)


Primary 30-day cohort ready [built from source CSVs and saved to primary_multimodal_30day_cohort.csv]: 77 assessments, 71 participants (CN=53, MCI=24)


## 3. Canonical model pipeline, participant aggregation, and metric helpers


In [9]:
# Canonical model pipeline, participant aggregation, and metric helpers
# (Shared across Primary Model Comparison, Calibration/DCA, and SHAP Explainability.)

from collections import defaultdict
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEEDS_23 = [11, 21, 42, 52, 62, 72, 82, 92, 102, 112]
PRIMARY_SEED_23 = 42
N_SPLITS_23 = 5
THRESHOLD_23 = 0.50

X_story23 = X_story22.copy()
X_ecog23 = X_ecog22.copy()
X_full23 = X_full22.copy()
y23 = np.asarray(y22).astype(int)
groups23 = np.asarray(groups22)

feature_sets23 = {
    "Storyteller only": list(X_story23.columns),
    "Full fusion": list(X_full23.columns),
    "Fusion without ECOGPT1": [col for col in X_full23.columns if col != "ECOGPT1"],
    "Fusion without ECOGPT1 and ECOGPT2": [
        col for col in X_full23.columns if col not in ["ECOGPT1", "ECOGPT2"]
    ],
    "Fusion without ECOGPT2": [col for col in X_full23.columns if col != "ECOGPT2"],
}


def create_model23():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                solver="liblinear",
                random_state=42,
            ),
        ),
    ])


def aggregate_participant_predictions23(y_true, participant_ids, probabilities):
    temp = pd.DataFrame({
        "PTID": participant_ids,
        "TARGET": y_true,
        "PROBABILITY": probabilities,
    })
    label_check = temp.groupby("PTID")["TARGET"].nunique()
    if (label_check > 1).any():
        raise ValueError("Inconsistent participant labels found during aggregation.")
    participant_predictions = (
        temp.groupby("PTID")
        .agg(
            TARGET=("TARGET", "first"),
            PROBABILITY=("PROBABILITY", "mean"),
            N_ASSESSMENTS=("TARGET", "size"),
        )
        .reset_index()
    )
    return participant_predictions


def calculate_metrics23(y_true, probabilities):
    predictions = (np.asarray(probabilities) >= THRESHOLD_23).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, predictions),
        "MCI_Precision": precision_score(y_true, predictions, zero_division=0),
        "MCI_Recall_Sensitivity": recall_score(y_true, predictions, zero_division=0),
        "Specificity": specificity,
        "MCI_F1": f1_score(y_true, predictions, zero_division=0),
        "Macro_F1": f1_score(y_true, predictions, average="macro", zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }
